# 10. 성과 그룹별 속성 역분석

**분석 목적:** `performance_grade`를 먼저 기준으로 삼고, 각 성과 그룹이 어떤 태그·장르·가격대 속성을 공유하는지 역방향으로 분석한다.

**핵심 질문:** 흥행권, 숨겨진 가능성, 미반응 게임은 어떤 태그·장르·가격대 조합에서 자주 나타나는가?

**사용 데이터:** `data/preprocessed/steam_indie_games_graded.csv`

## 분석 흐름

1. `performance_grade`를 발표용 성과 그룹으로 재분류
2. 성과 그룹별 상위 태그 비율 비교
3. 태그 특이도(Lift) 분석
4. 장르 × 성과 그룹 교차 히트맵
5. 성과 그룹별 가격 분포 비교
6. 전략적 해석 정리

In [1]:
from pathlib import Path
import ast
import json

import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

## 1. 데이터 로드

이미 `03_performance_grade.ipynb`에서 생성한 `steam_indie_games_graded.csv`를 사용한다. 이 파일에는 리뷰 규모(`scale_grade`), 만족도(`satisfaction_grade`), 종합 성과 등급(`performance_grade`)이 포함되어 있다.

In [2]:
DATA_PATH = Path("../../../data/preprocessed/steam_indie_games_graded.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "steam_indie_games_graded.csv를 찾을 수 없습니다. "
        "data/preprocessed/ 경로를 확인하세요."
    )

games = pd.read_csv(DATA_PATH)
games["release_date"] = pd.to_datetime(games["release_date"], errors="coerce")
games["release_year"] = games["release_date"].dt.year

# 09번 노트북의 집중 분석 기간과 맞춘다.
games = games.query("2023 <= release_year <= 2025").copy()
games["release_year"] = games["release_year"].astype(int)

print(f"분석 대상 게임 수: {len(games):,}")
print(f"출시 연도 범위: {games['release_year'].min()}~{games['release_year'].max()}")

games[["appid", "name", "release_year", "total_reviews", "positive_rate", "performance_grade"]].head()

분석 대상 게임 수: 8,825
출시 연도 범위: 2023~2025


,appid,name,release_year,total_reviews,positive_rate,performance_grade
0,226620,Desktop Dungeons,2023,2276,84.007030,high_high
1,230210,ASYLUM,2025,348,87.068966,mid_high
2,251570,7 Days to Die,2024,370046,88.607633,high_high
3,252190,Defender's Quest 2: Mists of Ruin,2025,255,61.568627,mid_low
4,269770,Secrets of Grindea,2024,8270,89.334946,high_high


## 2. 성과 그룹 재정의

`performance_grade` 9개를 그대로 비교하면 발표에서 해석이 복잡해진다. 따라서 퍼널의 마지막 단계인 `흥행권 진입`과 연결되도록 성과 유형을 5개 그룹으로 재분류한다.

| 성과 그룹 | 포함 등급 | 해석 |
|---|---|---|
| 흥행권 | `high_high`, `high_mid` | 리뷰 수가 많고 만족도(긍정률)도 높은 성공한 게임 |
| 숨겨진 가능성 | `mid_high`, `low_high` | 만족도는 높지만, 리뷰 수가 적어 확산되지 못한 게임 |
| 호불호/외면 | `high_low`, `mid_low` | 반응(리뷰)은 어느 정도 있으나 만족도가 낮은 게임 |
| 평범/미노출 | `mid_mid`, `low_mid` | 만족도도 중간, 확산도 안 된 게임 |
| 미반응 | `low_low` | 리뷰 수도 없고 만족도도 낮은 게임 |

In [3]:
PERFORMANCE_GROUP_MAP = {
    "high_high": "흥행권",
    "high_mid": "흥행권",
    "mid_high": "숨겨진 가능성",
    "low_high": "숨겨진 가능성",
    "high_low": "호불호/외면",
    "mid_low": "호불호/외면",
    "mid_mid": "평범/미노출",
    "low_mid": "평범/미노출",
    "low_low": "미반응",
}

GROUP_ORDER = ["흥행권", "숨겨진 가능성", "호불호/외면", "평범/미노출", "미반응"]
GROUP_COLOR = {
    "흥행권": "#2d6a4f",
    "숨겨진 가능성": "#4C72B0",
    "호불호/외면": "#DD8452",
    "평범/미노출": "#8C8C8C",
    "미반응": "#C44E52",
}

games["performance_group"] = games["performance_grade"].map(PERFORMANCE_GROUP_MAP)
games["performance_group"] = pd.Categorical(
    games["performance_group"],
    categories=GROUP_ORDER,
    ordered=True,
)

group_counts = (
    games.groupby("performance_group", observed=True)
    .agg(game_count=("appid", "nunique"), median_reviews=("total_reviews", "median"), median_positive_rate=("positive_rate", "median"))
    .reset_index()
)
group_counts["ratio"] = group_counts["game_count"] / group_counts["game_count"].sum() * 100

group_counts.round(2)

,performance_group,game_count,median_reviews,median_positive_rate,ratio
0,흥행권,999,1417.0,89.47,11.32
1,숨겨진 가능성,5421,32.0,92.86,61.43
2,호불호/외면,496,137.0,62.68,5.62
3,평범/미노출,1091,48.0,75.00,12.36
4,미반응,818,19.0,58.33,9.27


In [4]:
fig = px.bar(
    group_counts,
    x="performance_group",
    y="game_count",
    color="performance_group",
    color_discrete_map=GROUP_COLOR,
    text="ratio",
    title="성과 그룹별 게임 수 분포",
    labels={"performance_group": "성과 그룹", "game_count": "게임 수"},
    category_orders={"performance_group": GROUP_ORDER},
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(template="plotly_white", showlegend=False, width=900, height=480)
fig.show()

## 3. 속성 파싱

태그는 Steam 태그 투표수 기준 상위 5개만 사용한다. 모든 태그를 사용하면 희소 태그가 과도하게 많아지고, 각 게임의 핵심 포지셔닝이 흐려질 수 있다.

In [5]:
def parse_genres(value: str) -> list[str]:
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in str(value).split(",") if item.strip()]


def parse_tags(value: str) -> dict[str, int]:
    if pd.isna(value):
        return {}
    try:
        parsed = json.loads(value)
        if isinstance(parsed, dict):
            return {str(tag): int(score) for tag, score in parsed.items()}
    except (json.JSONDecodeError, TypeError, ValueError):
        return {}
    return {}


def top_tags(tag_dict: dict[str, int], n: int = 5) -> list[str]:
    return sorted(tag_dict, key=tag_dict.get, reverse=True)[:n]


games["genre_list"] = games["genres"].apply(parse_genres)
games["tag_dict"] = games["tags"].apply(parse_tags)
games["top_tags"] = games["tag_dict"].apply(lambda tags: top_tags(tags, n=5))

print(f"장르 보유 게임 수: {(games['genre_list'].map(len) > 0).sum():,}")
print(f"상위 태그 보유 게임 수: {(games['top_tags'].map(len) > 0).sum():,}")

장르 보유 게임 수: 8,825
상위 태그 보유 게임 수: 8,825


## 4. 성과 그룹별 상위 태그 비율

단순 빈도 대신 **성과 그룹 내 등장 비율**을 사용한다. 예를 들어 `흥행권` 그룹에서 특정 태그가 20% 등장했다면, 흥행권 게임 5개 중 1개가 그 태그를 핵심 태그로 가진다는 의미다.

In [6]:
tag_df = (
    games[games["top_tags"].map(len) > 0]
    .explode("top_tags")
    .rename(columns={"top_tags": "tag"})
)
tag_df["tag"] = tag_df["tag"].astype(str).str.strip()
tag_df = tag_df[tag_df["tag"] != ""].copy()

GROUP_MIN_TAG_GAMES = 10
TOP_N_PER_GROUP = 10

group_sizes = games.groupby("performance_group", observed=True)["appid"].nunique().rename("group_games")

group_tag_stats = (
    tag_df.groupby(["performance_group", "tag"], observed=True)
    .agg(tag_games=("appid", "nunique"))
    .reset_index()
    .merge(group_sizes.reset_index(), on="performance_group", how="left")
)
group_tag_stats["tag_rate"] = group_tag_stats["tag_games"] / group_tag_stats["group_games"] * 100

top_group_tags = (
    group_tag_stats[group_tag_stats["tag_games"] >= GROUP_MIN_TAG_GAMES]
    .sort_values(["performance_group", "tag_rate"], ascending=[True, False])
    .groupby("performance_group", observed=True)
    .head(TOP_N_PER_GROUP)
    .copy()
)

top_group_tags.head(20).round(2)

,performance_group,tag,tag_games,group_games,tag_rate
16,흥행권,Adventure,147,999,14.71
260,흥행권,Simulation,141,999,14.11
228,흥행권,RPG,125,999,12.51
11,흥행권,Action,114,999,11.41
273,흥행권,Strategy,104,999,10.41
47,흥행권,Casual,103,999,10.31
141,흥행권,Horror,100,999,10.01
104,흥행권,Exploration,90,999,9.01
211,흥행권,Pixel Graphics,89,999,8.91
261,흥행권,Singleplayer,89,999,8.91


In [7]:
fig = px.bar(
    top_group_tags.sort_values(["performance_group", "tag_rate"], ascending=[True, True]),
    x="tag_rate",
    y="tag",
    color="performance_group",
    facet_col="performance_group",
    facet_col_wrap=2,
    orientation="h",
    color_discrete_map=GROUP_COLOR,
    title="성과 그룹별 상위 태그 비율",
    labels={"tag_rate": "그룹 내 등장 비율(%)", "tag": "태그", "performance_group": "성과 그룹"},
    category_orders={"performance_group": GROUP_ORDER},
    hover_data={"tag_games": ":,", "group_games": ":,"},
)
fig.update_layout(template="plotly_white", width=1100, height=900, showlegend=False)
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.show()

## 5. 태그 특이도(Lift) 분석

Lift는 특정 성과 그룹에서 어떤 태그가 전체 평균보다 더 자주 등장하는지 보여준다.

```text
lift = 그룹 내 태그 등장 비율 / 전체 태그 등장 비율
```

- `lift > 1`: 해당 그룹에서 과대표현
- `lift < 1`: 해당 그룹에서 과소표현

희소 태그의 과대 해석을 줄이기 위해 전체 등장 게임 수와 그룹 내 등장 게임 수에 최소 기준을 둔다.

In [8]:
OVERALL_MIN_TAG_GAMES = 50
GROUP_MIN_TAG_GAMES_FOR_LIFT = 10

overall_games = games["appid"].nunique()
overall_tag_stats = (
    tag_df.groupby("tag")
    .agg(overall_tag_games=("appid", "nunique"))
    .reset_index()
)
overall_tag_stats["overall_tag_rate"] = overall_tag_stats["overall_tag_games"] / overall_games * 100

lift_df = group_tag_stats.merge(overall_tag_stats, on="tag", how="left")
lift_df = lift_df[
    (lift_df["overall_tag_games"] >= OVERALL_MIN_TAG_GAMES)
    & (lift_df["tag_games"] >= GROUP_MIN_TAG_GAMES_FOR_LIFT)
].copy()
lift_df["lift"] = lift_df["tag_rate"] / lift_df["overall_tag_rate"]
lift_df["rate_diff"] = lift_df["tag_rate"] - lift_df["overall_tag_rate"]

signature_tags = (
    lift_df.sort_values(["performance_group", "lift", "rate_diff"], ascending=[True, False, False])
    .groupby("performance_group", observed=True)
    .head(10)
    .copy()
)

signature_tags[["performance_group", "tag", "tag_games", "overall_tag_games", "tag_rate", "overall_tag_rate", "lift", "rate_diff"]].round(2)

,performance_group,tag,tag_games,overall_tag_games,tag_rate,overall_tag_rate,lift,rate_diff
35,흥행권,Boomer Shooter,18,62,1.80,0.70,2.56,1.10
198,흥행권,Open World,42,150,4.20,1.70,2.47,2.50
180,흥행권,Multiplayer,74,278,7.41,3.15,2.35,4.26
243,흥행권,Roguelike Deckbuilder,31,117,3.10,1.33,2.34,1.78
57,흥행권,Co-op,37,144,3.70,1.63,2.27,2.07
29,흥행권,Base-Building,31,121,3.10,1.37,2.26,1.73
197,흥행권,Online Co-Op,47,185,4.70,2.10,2.24,2.61
26,흥행권,Automation,17,67,1.70,0.76,2.24,0.94
135,흥행권,Hentai,13,54,1.30,0.61,2.13,0.69
112,흥행권,Farming Sim,25,104,2.50,1.18,2.12,1.32


In [9]:
fig = px.scatter(
    signature_tags,
    x="rate_diff",
    y="lift",
    color="performance_group",
    size="tag_games",
    text="tag",
    color_discrete_map=GROUP_COLOR,
    title="성과 그룹별 시그니처 태그: Lift × 비율 차이",
    labels={
        "rate_diff": "전체 대비 그룹 내 등장 비율 차이(%p)",
        "lift": "Lift",
        "tag_games": "그룹 내 게임 수",
        "performance_group": "성과 그룹",
    },
    category_orders={"performance_group": GROUP_ORDER},
    hover_data={"tag_rate": ":.1f", "overall_tag_rate": ":.1f", "tag_games": ":,"},
)
fig.add_hline(y=1, line_dash="dash", line_color="#666666")
fig.add_vline(x=0, line_dash="dash", line_color="#666666")
fig.update_traces(textposition="top center", marker=dict(opacity=0.75))
fig.update_layout(template="plotly_white", width=1050, height=650)
fig.show()

In [10]:
heatmap_tags = signature_tags["tag"].drop_duplicates().tolist()
lift_heatmap_data = lift_df[lift_df["tag"].isin(heatmap_tags)].copy()

lift_pivot = (
    lift_heatmap_data.pivot_table(
        index="tag",
        columns="performance_group",
        values="lift",
        aggfunc="max",
        observed=True,
    )
    .reindex(columns=GROUP_ORDER)
    .fillna(0)
)

fig = px.imshow(
    lift_pivot,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=1,
    title="성과 그룹별 시그니처 태그 Lift 히트맵",
    labels={"x": "성과 그룹", "y": "태그", "color": "Lift"},
)
fig.update_layout(template="plotly_white", width=950, height=max(520, 22 * len(lift_pivot)))
fig.show()

## 6. 장르 × 성과 그룹 히트맵

장르는 다중 장르를 모두 반영하기 위해 explode 방식으로 집계한다. `Indie`는 모든 게임에 가까운 공통 속성이므로 비교에서 제외한다.

In [11]:
genre_df = (
    games[games["genre_list"].map(len) > 0]
    .explode("genre_list")
    .rename(columns={"genre_list": "genre"})
)
genre_df["genre"] = genre_df["genre"].astype(str).str.strip()
genre_df = genre_df[(genre_df["genre"] != "") & (genre_df["genre"] != "Indie")].copy()

MIN_GAMES_PER_GENRE = 30
valid_genres = (
    genre_df.groupby("genre")["appid"].nunique()
    .pipe(lambda counts: counts[counts >= MIN_GAMES_PER_GENRE].index.tolist())
)
genre_df = genre_df[genre_df["genre"].isin(valid_genres)].copy()

genre_group_counts = (
    genre_df.groupby(["genre", "performance_group"], observed=True)
    .agg(game_count=("appid", "nunique"))
    .reset_index()
)
genre_totals = genre_group_counts.groupby("genre")["game_count"].transform("sum")
genre_group_counts["group_ratio"] = genre_group_counts["game_count"] / genre_totals * 100

pivot_genre_group = (
    genre_group_counts.pivot(index="genre", columns="performance_group", values="group_ratio")
    .reindex(columns=GROUP_ORDER)
    .fillna(0)
)

pivot_genre_group.round(1)

performance_group,흥행권,숨겨진 가능성,호불호/외면,평범/미노출,미반응
genre,,,,,
Action,11.5,59.9,6.4,12.3,9.8
Adventure,11.8,59.4,5.6,13.0,10.2
Casual,8.9,66.2,4.8,10.7,9.3
RPG,16.1,53.1,7.8,14.1,8.9
Racing,7.6,59.6,5.1,14.2,13.5
Simulation,15.2,47.4,9.1,15.1,13.2
Sports,5.4,63.7,4.1,14.6,12.1
Strategy,14.9,55.2,7.5,13.9,8.4


In [12]:
fig = px.imshow(
    pivot_genre_group,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="YlGnBu",
    title="장르별 성과 그룹 비율 (다중 장르 중복 집계)",
    labels={"x": "성과 그룹", "y": "장르", "color": "비율(%)"},
)
fig.update_layout(template="plotly_white", width=950, height=max(480, 36 * len(pivot_genre_group)))
fig.show()

## 7. 성과 그룹별 가격 포지셔닝

가격은 게임 속성이라기보다 출시 전략 변수다. 따라서 “흥행권 게임의 정답 가격”을 찾기보다는, 성과 그룹별 가격 포지셔닝 차이를 확인하는 용도로 해석한다.

In [13]:
price_games = games.copy()
price_games["price"] = pd.to_numeric(price_games["price"], errors="coerce")
price_games = price_games.dropna(subset=["price", "performance_group"]).copy()
price_games = price_games[(price_games["price"] > 0) & (price_games["price"] <= 60)].copy()

PRICE_BINS = [0, 5, 10, 15, 20, 30, 60]
PRICE_LABELS = ["~$5", "$5~10", "$10~15", "$15~20", "$20~30", "$30~60"]
price_games["price_range"] = pd.cut(price_games["price"], bins=PRICE_BINS, labels=PRICE_LABELS, right=True)

price_summary = (
    price_games.groupby("performance_group", observed=True)
    .agg(
        game_count=("appid", "nunique"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
        q1_price=("price", lambda s: s.quantile(0.25)),
        q3_price=("price", lambda s: s.quantile(0.75)),
    )
    .reset_index()
)

price_summary.round(2)

,performance_group,game_count,median_price,avg_price,q1_price,q3_price
0,흥행권,991,12.99,14.49,7.14,19.99
1,숨겨진 가능성,5373,5.99,7.67,2.99,9.99
2,호불호/외면,482,9.99,11.29,4.99,15.99
3,평범/미노출,1067,6.99,9.04,3.99,12.99
4,미반응,795,4.99,6.78,1.99,9.99


In [14]:
fig = px.box(
    price_games,
    x="performance_group",
    y="price",
    color="performance_group",
    color_discrete_map=GROUP_COLOR,
    category_orders={"performance_group": GROUP_ORDER},
    points=False,
    title="성과 그룹별 가격 분포",
    labels={"performance_group": "성과 그룹", "price": "가격 (USD)"},
)
fig.update_layout(template="plotly_white", showlegend=False, width=950, height=520)
fig.update_yaxes(tickprefix="$")
fig.show()

In [15]:
price_group_counts = (
    price_games.groupby(["performance_group", "price_range"], observed=True)
    .agg(game_count=("appid", "nunique"))
    .reset_index()
)
price_group_total = price_group_counts.groupby("performance_group", observed=True)["game_count"].transform("sum")
price_group_counts["ratio"] = price_group_counts["game_count"] / price_group_total * 100

fig = px.bar(
    price_group_counts,
    x="performance_group",
    y="ratio",
    color="price_range",
    title="성과 그룹별 가격대 구성 비율",
    labels={"performance_group": "성과 그룹", "ratio": "비율(%)", "price_range": "가격대"},
    category_orders={"performance_group": GROUP_ORDER, "price_range": PRICE_LABELS},
    text="ratio",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="inside")
fig.update_layout(template="plotly_white", width=950, height=520, yaxis=dict(ticksuffix="%"))
fig.show()

## 8. 종합 해석 가이드

- `흥행권`에서 lift와 비율 차이가 모두 높은 태그는 최근 흥행권 게임의 시그니처 포지셔닝 후보로 볼 수 있다.
- `숨겨진 가능성`은 만족도는 높지만 확산이 부족한 그룹이다. 흥행권과 공통 태그가 많다면 마케팅·노출 전략 차이를 추가로 의심할 수 있고, 다른 태그가 많다면 니치 장르 특성으로 해석할 수 있다.
- `미반응`에서 과대표현되는 태그·장르는 출시 전 시장성 검토나 차별화 전략이 더 필요한 영역으로 볼 수 있다.
- 가격은 독립적인 성공 원인이라기보다 장르·태그와 함께 해석해야 한다. 같은 가격대라도 장르별 기대 콘텐츠 규모가 다르기 때문이다.